# Specialiserede Modeller — 4: Clustering — mønstre UDEN labels

Alt, hvad I har trænet indtil nu, havde en **facitliste**: giftig/spiselig, syg/rask,
hvilket ciffer. Men i den virkelige verden har data tit INGEN labels — bare rå
observationer. Kan en model overhovedet lære noget så? Ja: den kan finde **grupper**.
Det hedder **unsupervised learning**, og dagens algoritme er klassikeren **k-means**.

Scenariet: I er dataanalytikere for et butikscenter. I har 200 kunder med årsindkomst
og en "spending score" (1-100: hvor meget shopper de?). Chefen spørger: **"hvilke
kundetyper har vi?"** — og der findes ingen facitliste. Værsgo.

> Denne notebook er selvkørende og kræver kun viden fra Intro-ML — du kan tage emnets notebooks i den rækkefølge, du vil. Der er med vilje flere opgaver, end du kan nå: opgaver mærket **(ekstra)** er til de hurtige, og opgaver mærket **(find fejlen)** indeholder en bevidst fejl, som skal findes og rettes.

## Setup

In [ ]:
# Henter kunde-data fra GitHub (Plan B: upload Mall_Customers.csv via mappeikonet i Colab)
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Specialiserede-Modeller/data/Mall_Customers.csv

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Specialiserede-Modeller/hjaelpefunktioner.py

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hjaelpefunktioner import plot_kmeans_skridt

np.random.seed(42)

df = pd.read_csv("Mall_Customers.csv")
df.head()

# 7: k-means — find klumperne

Lad os først SE på kunderne. Vi plotter årsindkomst mod spending score:

In [ ]:
X_raa = df[["Annual Income (k$)", "Spending Score (1-100)"]].values.astype("float32")

plt.figure(figsize=(7, 5))
plt.scatter(X_raa[:, 0], X_raa[:, 1], alpha=0.7)
plt.xlabel("årsindkomst (tusind $)")
plt.ylabel("spending score")
plt.title("200 kunder — kan I se grupperne?")
plt.show()

Kig godt — der ER tydelige klumper (de fleste ser 5). Men vi vil ikke sidde og tegne
cirkler i hånden på 200 kunder... og slet ikke på 2 millioner. Vi vil have en algoritme.

## Algoritmen: så simpel, at det næsten er snyd

**k-means** finder $k$ klynger sådan her:

1. Placér $k$ **centre** tilfældigt (fx oven på $k$ tilfældige datapunkter).
2. **Tildel**: hvert punkt hører til det nærmeste center.
3. **Flyt**: hvert center flytter til gennemsnittet (middelpunktet) af sine punkter.
4. Gentag 2-3, indtil intet flytter sig.

Ingen gradienter, ingen tabsfunktion at differentiere — bare afstande og gennemsnit.
Først standardiserer vi (afstande på tværs af enheder kræver fælles skala — Intro-ML-
lektionen igen!), og så kører vi algoritmen SYNLIGT, trin for trin:

In [ ]:
X = (X_raa - X_raa.mean(axis=0)) / X_raa.std(axis=0)     # standardisering

k = 5
rng = np.random.default_rng(42)
centre = X[rng.choice(len(X), size=k, replace=False)].copy()   # trin 1: start på k tilfældige kunder

centre_historik = []
tildelinger_historik = []

for iteration in range(6):
    # trin 2: afstand fra hvert punkt til hvert center → vælg det nærmeste
    afstande = np.linalg.norm(X[:, None, :] - centre[None, :, :], axis=2)
    tildeling = afstande.argmin(axis=1)

    centre_historik.append(centre.copy())
    tildelinger_historik.append(tildeling.copy())

    # trin 3: flyt hvert center til gennemsnittet af dets punkter
    for j in range(k):
        centre[j] = X[tildeling == j].mean(axis=0)

plot_kmeans_skridt(X, centre_historik, tildelinger_historik)

Følg de sorte kryds (centrene) fra plot til plot: de vandrer ind i hver sin klump og
falder til ro — algoritmen er typisk færdig på en håndfuld iterationer. (Linjen med
`np.linalg.norm` regner alle 200×5 afstande på én gang — det er numpy-broadcasting og
må godt føles som sort magi; pointen er trin 2's "vælg nærmeste center".)

## Det professionelle værktøj: sklearn

Som altid i sklearn: samme mønster som træerne — bare uden `y`, for der ER ingen labels:

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=5, n_init=10, random_state=42)
kmeans.fit(X)                                  # bemærk: KUN X — ingen y!

plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], c=kmeans.labels_, cmap="tab10", alpha=0.8)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            marker="X", s=250, color="black", edgecolors="white")
plt.xlabel("indkomst (standardiseret)")
plt.ylabel("spending score (standardiseret)")
plt.title("sklearn's KMeans, k = 5")
plt.show()

## Fra klynger til KUNDETYPER

Tallene 0-4 betyder ingenting i sig selv — fortolkningen er VORES arbejde. Vi hænger
klyngenumrene på kunderne og regner gennemsnit pr. klynge:

In [ ]:
df["klynge"] = kmeans.labels_

profil = df.groupby("klynge")[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].mean().round(1)
profil["antal"] = df["klynge"].value_counts().sort_index()
profil

Prøv at give hver klynge et navn ud fra tallene — fx "velhavende storshoppere"
(høj indkomst + høj score) eller "forsigtige velhavere" (høj indkomst + lav score).
Det er præcis den slags segmenter, marketingafdelinger bygger kampagner på.

## Hvordan vælger man k? Albue-metoden

k-means svarer ALTID med præcis $k$ klynger — også hvis $k$ er helt skævt. Et
pejlemærke: mål den samlede afstand fra punkterne til deres centre (kaldes **inertia**
— sklearn regner den ud for os) for forskellige $k$, og se hvor kurven "knækker" som
en albue: dér holder ekstra klynger op med at hjælpe for alvor.

In [ ]:
inertias = []
k_vaerdier = range(1, 11)
for k_test in k_vaerdier:
    km = KMeans(n_clusters=k_test, n_init=10, random_state=42)
    km.fit(X)
    inertias.append(km.inertia_)

plt.plot(k_vaerdier, inertias, "o-")
plt.xlabel("k (antal klynger)")
plt.ylabel("inertia (samlet afstand til centre)")
plt.title("Albue-metoden — find knækket")
plt.show()

### Opgaver

##### Opgave 7.1
Kør trin-for-trin-algoritmen igen, men med en anden start:
`rng = np.random.default_rng(0)` (og prøv flere). Ender centrene samme sted hver gang?
Hvad minder det jer om fra gradient descent-notebooken i Intro-ML?

In [ ]:
rng = np.random.default_rng(0)      # ← prøv 0, 1, 7 ...
centre = X[rng.choice(len(X), size=5, replace=False)].copy()
centre_historik, tildelinger_historik = [], []
for iteration in range(6):
    afstande = np.linalg.norm(X[:, None, :] - centre[None, :, :], axis=2)
    tildeling = afstande.argmin(axis=1)
    centre_historik.append(centre.copy())
    tildelinger_historik.append(tildeling.copy())
    for j in range(5):
        centre[j] = X[tildeling == j].mean(axis=0)
plot_kmeans_skridt(X, centre_historik, tildelinger_historik)

##### Opgave 7.2
Kør sklearn-KMeans med **k = 2, 3 og 8** og plot resultaterne. Beskriv med jeres egne
ord, hvad algoritmen "gør ved" kunderne, når k er for lille — og når k er for stor.

In [ ]:
k_test = 2       # ← prøv 2, 3, 8
km = KMeans(n_clusters=k_test, n_init=10, random_state=42)
km.fit(X)
plt.scatter(X[:, 0], X[:, 1], c=km.labels_, cmap="tab10", alpha=0.8)
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
            marker="X", s=250, color="black", edgecolors="white")
plt.title(f"k = {k_test}")
plt.show()

##### Opgave 7.3
Udfyld albue-loopet (fit + gem `inertia_`) og aflæs: hvor sidder albuen for
kundedataene? Passer det med det antal klumper, I selv så i det første scatter-plot?

In [ ]:
inertias = []
for k_test in range(1, 11):
    km = KMeans(n_clusters=k_test, n_init=10, random_state=42)
    ...
    inertias.append(...)

plt.plot(range(1, 11), inertias, "o-")
plt.xlabel("k")
plt.ylabel("inertia")
plt.show()

##### Opgave 7.4
Hvad sker der uden standardisering? Forestil jer, at indkomsten stod i **kroner** i
stedet for tusind dollars — kør cellen og se på klyngerne. Hvorfor ignorerer k-means
pludselig spending score fuldstændigt?

In [ ]:
X_kroner = X_raa.copy()
X_kroner[:, 0] = X_kroner[:, 0] * 7000        # tusind $ → kroner (ca.)

km = KMeans(n_clusters=5, n_init=10, random_state=42)
km.fit(X_kroner)

plt.scatter(X_kroner[:, 0], X_kroner[:, 1], c=km.labels_, cmap="tab10", alpha=0.8)
plt.xlabel("årsindkomst (kroner)")
plt.ylabel("spending score")
plt.title("k-means på u-standardiserede data")
plt.show()

##### Opgave 7.5
Udfyld `groupby`-profilen, og giv hver af de 5 klynger et navn (à la "velhavende
storshoppere"). Hvilken klynge ville I sende luksus-reklamer til — og hvilken ville I
sende rabatkuponer?

In [ ]:
df["klynge"] = kmeans.labels_
profil = df.groupby(...)[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].mean().round(1)
profil["antal"] = df["klynge"].value_counts().sort_index()
profil
# jeres navne:
# klynge 0: ...
# klynge 1: ...

##### Opgave 7.6 (find fejlen)
En kammerat har taget "alle talkolonnerne" med i clusteringen — og klyngerne er blevet
mærkeligt meningsløse striber. Kig på kolonnelisten: hvilken kolonne har intet at gøre i
en afstandsberegning, og hvorfor ødelægger den alt?

In [ ]:
X_fejl = df[["CustomerID", "Annual Income (k$)", "Spending Score (1-100)"]].values.astype("float32")
X_fejl = (X_fejl - X_fejl.mean(axis=0)) / X_fejl.std(axis=0)

km = KMeans(n_clusters=5, n_init=10, random_state=42)
km.fit(X_fejl)

plt.scatter(X_fejl[:, 1], X_fejl[:, 2], c=km.labels_, cmap="tab10", alpha=0.8)
plt.xlabel("indkomst (std)")
plt.ylabel("spending score (std)")
plt.title("hmm... klyngerne giver ikke mening")
plt.show()

##### Opgave 7.7
I supervised learning kunne vi måle accuracy mod en facitliste. Her er der ingen. Så:
hvordan ved vi overhovedet, om VORES klynger er "rigtige"? Kan to forskellige opdelinger
begge være "gode"? Diskutér — og find på mindst ét konkret tjek, man kunne lave.

*Skriv jeres svar her:* $\dots$

##### Opgave 7.8
Lav en helt anden segmentering: clustr på **Age** og **Spending Score** i stedet
(husk standardisering!). Hvilke nye grupper dukker op — og hvilket k foreslår
albue-metoden her?

In [ ]:
X2_raa = df[["Age", "Spending Score (1-100)"]].values.astype("float32")
X2 = ...                                     # standardisér!

km2 = KMeans(n_clusters=4, n_init=10, random_state=42)
km2.fit(X2)
plt.scatter(X2[:, 0], X2[:, 1], c=km2.labels_, cmap="tab10", alpha=0.8)
plt.xlabel("alder (std)")
plt.ylabel("spending score (std)")
plt.show()

##### Opgave 7.9
k-means' akilleshæl: kør den på månedataene fra Intro-ML (`make_moons`) og plot
resultatet. Hvorfor fejler den her, når jeres neurale netværk kunne? (Tænk på, hvad
"nærmeste center" betyder geometrisk.)

In [ ]:
from sklearn.datasets import make_moons

X_m, y_rigtige = make_moons(n_samples=400, noise=0.05, random_state=42)

km_m = KMeans(n_clusters=2, n_init=10, random_state=42)
km_m.fit(X_m)

fig, akser = plt.subplots(1, 2, figsize=(11, 4))
akser[0].scatter(X_m[:, 0], X_m[:, 1], c=y_rigtige, cmap="tab10", s=15)
akser[0].set_title("de RIGTIGE grupper")
akser[1].scatter(X_m[:, 0], X_m[:, 1], c=km_m.labels_, cmap="tab10", s=15)
akser[1].set_title("k-means' bud")
plt.show()

##### Opgave 7.10 (ekstra)
Fuld segmentering: clustr på alle tre features (**Age, Annual Income, Spending
Score**). Kør albue-metoden, vælg k, og lav klynge-profilen med `groupby`. Kan I stadig
give hver klynge et navn? (3D kan ikke plottes direkte — profilen ER jeres billede.)

In [ ]:
X3_raa = df[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].values.astype("float32")
X3 = (X3_raa - X3_raa.mean(axis=0)) / X3_raa.std(axis=0)

# albue → vælg k → fit → groupby-profil
...

##### Opgave 7.11 (ekstra)
sklearn's `inertia_` er bare summen af kvadrerede afstande fra hvert punkt til dets
center. Regn den selv for jeres k=5-model (udfyld), og tjek at I rammer sklearn's tal.

In [ ]:
km = KMeans(n_clusters=5, n_init=10, random_state=42)
km.fit(X)

total = 0.0
for j in range(5):
    punkter = X[km.labels_ == j]
    center = km.cluster_centers_[j]
    total = total + ...          # summen af kvadrerede afstande til centret

print("vores:  ", round(total, 2))
print("sklearn:", round(km.inertia_, 2))

##### Opgave 7.12
Butikscentret vil bruge jeres segmenter til målrettede tilbud og forskellige priser til
forskellige kundetyper. Diskutér: hvad kunne det bruges FORNUFTIGT til — og hvor går
den etiske grænse? (Tænk fx på klyngen "lav indkomst + høj spending score"...)

*Skriv jeres svar her:* $\dots$

# Godt gået!

I har nu lært jeres første unsupervised algoritme: k-means trin for trin, sklearn's
udgave, albue-metoden, fortolkning af klynger — og tre klassiske fælder (skalering,
meningsløse features, ikke-runde klynger).

**Værktøjskassen:** tabeldata → træer · billeder → CNN · data UDEN labels →
clustering · sekvenser → RNN (næste notebook!).